In [1]:
from vllm import LLM

# Create an LLM.
# You should pass task="classify" for classification models
model = LLM(
    model="jason9693/Qwen2.5-1.5B-apeach",
    task="classify",
    enforce_eager=True,
)

INFO 07-29 22:50:49 [__init__.py:244] Automatically detected platform cuda.
INFO 07-29 22:50:59 [config.py:3368] Downcasting torch.float32 to torch.float16.
INFO 07-29 22:50:59 [config.py:1472] Using max model len 131072
INFO 07-29 22:50:59 [arg_utils.py:1596] (Disabling) chunked prefill by default
INFO 07-29 22:50:59 [arg_utils.py:1599] (Disabling) prefix caching by default
WARNING 07-29 22:50:59 [cuda.py:102] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 07-29 22:50:59 [config.py:4601] Only "last" pooling supports chunked prefill and prefix caching; disabling both.
INFO 07-29 22:51:00 [core.py:526] Waiting for init message from front-end.
INFO 07-29 22:51:00 [core.py:69] Initializing a V1 LLM engine (v0.9.2) with config: model='jason9693/Qwen2.5-1.5B-apeach', speculative_config=None, tokenizer='jason9693/Qwen2.5-1.5B-apeach', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, over

[W729 22:51:01.526508051 socket.cpp:755] [c10d] The client socket cannot be initialized to connect to [::ffff:10.244.0.115]:35225 (errno: 97 - Address family not supported by protocol).


INFO 07-29 22:51:02 [gpu_model_runner.py:1775] Loading model from scratch...
INFO 07-29 22:51:02 [cuda.py:284] Using Flash Attention backend on V1 engine.
INFO 07-29 22:51:02 [weight_utils.py:292] Using model weights format ['*.safetensors']


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 07-29 22:51:08 [default_loader.py:272] Loading weights took 5.49 seconds
INFO 07-29 22:51:09 [gpu_model_runner.py:1801] Model loading took 2.9110 GiB and 6.189718 seconds
INFO 07-29 22:51:11 [gpu_worker.py:232] Available KV cache memory: 24.47 GiB
INFO 07-29 22:51:11 [kv_cache_utils.py:716] GPU KV cache size: 916,512 tokens
INFO 07-29 22:51:11 [kv_cache_utils.py:720] Maximum concurrency for 131,072 tokens per request: 6.99x
INFO 07-29 22:51:12 [core.py:172] init engine (profile, create kv cache, warmup model) took 3.05 seconds
INFO 07-29 22:51:12 [config.py:4601] Only "last" pooling supports chunked prefill and prefix caching; disabling both.


In [2]:
# Sample prompts.
prompts = [
    "Hello, my name is",
    "The president of the United States is",
    "The capital of France is",
    "The future of AI is",
]

# Generate logits. The output is a list of ClassificationRequestOutputs.
outputs = model.classify(prompts)

# Print the outputs.
for prompt, output in zip(prompts, outputs):
    probs = output.outputs.probs
    probs_trimmed = ((str(probs[:16])[:-1] +
                      ", ...]") if len(probs) > 16 else probs)
    print(f"Prompt: {prompt!r} | "
          f"Class Probabilities: {probs_trimmed} (size={len(probs)})")

Adding requests:   0%|          | 0/4 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/4 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: 'Hello, my name is' | Class Probabilities: [0.2706729769706726, 0.7293270230293274] (size=2)
Prompt: 'The president of the United States is' | Class Probabilities: [0.0026316740550100803, 0.997368335723877] (size=2)
Prompt: 'The capital of France is' | Class Probabilities: [0.002183780074119568, 0.9978162050247192] (size=2)
Prompt: 'The future of AI is' | Class Probabilities: [0.034293945878744125, 0.9657061100006104] (size=2)


In [1]:
import time
import requests
import pandas as pd

columns = ['author', 'body', 'ups', 'downs', 'score']

for author in ["spez", "kn0thing"]:
    url = f"http://www.reddit.com/user/{author}/comments/.json?limit=100"
    response = requests.get(url)
    while response.status_code != 200:
        response = requests.get(url)
        time.sleep(10)
    data = response.json()
    df = pd.DataFrame()
    for comment in data['data']['children']:
        df = pd.concat([df, pd.DataFrame([comment['data']])], ignore_index=True)
    df = df[columns].copy()
    df.to_csv(f"{author}.csv", index=False,quoting=2, sep=';')

In [5]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import GuidedDecodingParams

guided_decoding_params = GuidedDecodingParams(choice=["Aktiva", "Passiva", "GuV", "other"],)
sampling_params = SamplingParams(guided_decoding=guided_decoding_params, logprobs=1, temperature=0)


# Generate logits. The output is a list of ClassificationRequestOutputs.
outputs = model.classify(
    prompts,
    sampling_params
)

# Print the outputs.
for prompt, output in zip(prompts, outputs):
    probs = output.outputs.probs
    probs_trimmed = ((str(probs[:16])[:-1] +
                      ", ...]") if len(probs) > 16 else probs)
    print(f"Prompt: {prompt!r} | "
          f"Class Probabilities: {probs_trimmed} (size={len(probs)})")

TypeError: LLM.classify() takes 2 positional arguments but 3 were given